In [4]:
import os
from pathlib import Path

from dotenv import load_dotenv

workspace_root = Path.cwd().parent
env_paths = (
    Path.cwd() / ".env",
    workspace_root / ".env",
    workspace_root / "Langchain_Basics" / ".env",
)

for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded environment from: {env_path}")
        break
else:
    print("No .env file found. Create Agents/.env or workspace-root/.env.")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")

Loaded environment from: c:\Users\Girish Kulkarni\Downloads\LangChainTrainings\Langchain_Basics\.env
LangSmith tracing enabled for project: Firstproject


In [5]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
    reasoning=False,
)

In [ ]:
# Wikipedia tool with retry handling

import json
import time

from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=4000,
    )
)


def wikipedia_invoke_with_retry(query, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            return wikipedia.invoke(query)
        except (json.JSONDecodeError, ConnectionError, TimeoutError) as error:
            if attempt == max_attempts - 1:
                return f"Wikipedia request failed after {max_attempts} attempts: {error}"
            time.sleep(2 ** attempt)


tool_response = wikipedia_invoke_with_retry("What is the capital of India?")
tool_response

In [ ]:
# Simple DuckDuckGo search

from langchain_community.tools import DuckDuckGoSearchRun

duckduckgo_search = DuckDuckGoSearchRun()

search_result = duckduckgo_search.invoke("Where is the Eiffel Tower located?")
print(search_result)

In [ ]:
# Creatin custome tools

from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def substarct(a: int, b: int) -> int:
    """Add two numbers."""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Add two numbers."""
    return a * b

print(add.invoke({"a":10, "b": 20}))  # Example usage of the custom tool



In [ ]:
tools = [wikipedia, add, substarct, multiply]

list_of_tools = {tool.name: tool for tool in tools}

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "What is the capital of India?"
)

response

In [ ]:
# Execute the custom tools with LLM

from langchain_core.messages import SystemMessage, HumanMessage


query = "what is the capital of India? AND 2=4 = ?"

message = [ human_message := HumanMessage(content=query) ]

ai_message = llm_with_tools.invoke(query)

ai_message.tool_calls




In [ ]:
# Execute the tools requested by the LLM

for tool_call in ai_message.tool_calls:
    tool_name = tool_call["name"].lower()
    tool_args = tool_call["args"]
    execute_tool = list_of_tools[tool_name]

    if tool_name == "wikipedia":
        tool_result = wikipedia_invoke_with_retry(**tool_args)
    else:
        tool_result = execute_tool.invoke(tool_args)

    print(f"{tool_name}: {tool_result}")